# Chapter 3 — Normalized Cut

Spectral Clustering minimizes the raw cut size, which tends to isolate small or weakly connected nodes.  
**Normalized Cut** (Shi & Malik, 2000) fixes this by normalizing the cut relative to the total connectivity of each partition — penalizing unbalanced splits.

This chapter covers:
- The Ncut criterion and why it improves on plain min-cut
- Single partition via the Fiedler vector
- Recursive partitioning for multi-way cuts

---

## Part 1 — Initialization

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.linalg import eigh

In [ ]:
def build_graph(N, M, seed=42, weight=1.0):
    """
    Build a random weighted graph and return G with all required matrices.

    Parameters
    ----------
    N      : int   — number of nodes
    M      : int   — number of edges
    seed   : int   — random seed
    weight : float — fixed edge weight (thesis uses unit weights)

    Returns
    -------
    G   : networkx Graph
    W   : weight matrix (N x N)
    D   : degree matrix (N x N diagonal)
    pos : node layout for plotting
    """
    G = nx.gnm_random_graph(N, M, seed=seed)
    for u, v in G.edges():
        G[u][v]['weight'] = weight
    pos = nx.spring_layout(G, seed=seed)
    W = nx.to_numpy_array(G, weight='weight')
    D = np.diag(W.sum(axis=1))
    return G, W, D, pos

---
## Part 2 — Normalized Cut Algorithm

### Theory

Given a partition of graph $G$ into sets $A$ and $B$, define:

$$\text{cut}(A, B) = \sum_{u \in A,\, v \in B} w(u,v)$$

$$\text{assoc}(A, V) = \sum_{u \in A,\, t \in V} w(u,t)$$

The **Normalized Cut** criterion is:

$$\text{Ncut}(A, B) = \frac{\text{cut}(A,B)}{\text{assoc}(A,V)} + \frac{\text{cut}(B,A)}{\text{assoc}(B,V)}$$

Minimizing Ncut is NP-hard in the discrete case. The continuous relaxation leads to a **generalized eigenvalue problem**:

$$(D - W)\,\mathbf{y} = \lambda D\,\mathbf{y}$$

which is equivalent to computing eigenvectors of the **normalized Laplacian** $L_N = D^{-1/2}(D-W)D^{-1/2}$.  
The partition is recovered from the **Fiedler vector** — the eigenvector corresponding to the 2nd smallest eigenvalue — via:

$$\mathbf{y}_1 = D^{-1/2}\,\mathbf{z}_1, \quad A = \{i : y_1(i) \geq 0\},\quad B = \{i : y_1(i) < 0\}$$

In [ ]:
def normalized_cut(G, W, D):
    """
    Single normalized cut partition using the Fiedler vector.

    Parameters
    ----------
    G : networkx Graph
    W : weight matrix (N x N)
    D : degree matrix (N x N diagonal)

    Returns
    -------
    set_A      : node indices in partition A
    set_B      : node indices in partition B
    metrics    : dict with cut, assoc, and ncut values
    """
    N = len(W)
    degrees = np.diag(D)

    # Normalized Laplacian: L_sym = D^{-1/2} (D - W) D^{-1/2}
    D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees))
    L_sym = np.eye(N) - D_inv_sqrt @ W @ D_inv_sqrt

    # Fiedler vector: eigenvector of 2nd smallest eigenvalue
    eigvals, eigvecs = eigh(L_sym)
    z1 = eigvecs[:, 1]
    y1 = D_inv_sqrt @ z1  # back-transform

    # Partition by sign of y1
    set_A = np.where(y1 >= 0)[0]
    set_B = np.where(y1 < 0)[0]

    # Compute metrics
    assoc_A = np.sum(W[set_A, :])
    assoc_B = np.sum(W[set_B, :])
    cut_AB  = np.sum(W[np.ix_(set_A, set_B)])
    ncut    = (cut_AB / assoc_A) + (cut_AB / assoc_B) if assoc_A > 0 and assoc_B > 0 else float('inf')

    metrics = {
        'assoc_A': assoc_A,
        'assoc_B': assoc_B,
        'cut_AB':  cut_AB,
        'ncut':    round(ncut, 4),
        'fiedler_vector': y1
    }
    return set_A, set_B, metrics

### Example 1 — 12 nodes, 21 edges

In [ ]:
G1, W1, D1, pos1 = build_graph(N=12, M=21, seed=42)
A1, B1, m1 = normalized_cut(G1, W1, D1)

colors1 = ['#e67e22' if n in A1 else '#27ae60' for n in G1.nodes()]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

nx.draw(G1, pos1, with_labels=True, node_color='steelblue',
        edge_color='gray', node_size=500, font_color='white',
        font_weight='bold', ax=axes[0])
axes[0].set_title('Original Graph (N=12, M=21)', fontsize=11)

nx.draw(G1, pos1, with_labels=True, node_color=colors1,
        edge_color='gray', node_size=500, font_color='white',
        font_weight='bold', ax=axes[1])
axes[1].set_title('After Normalized Cut', fontsize=11)

plt.tight_layout()
plt.show()

print("Partition A (orange):", sorted(A1.tolist()))
print("Partition B (green) :", sorted(B1.tolist()))
print(f"\nassoc(A,V) = {m1['assoc_A']:.2f}")
print(f"assoc(B,V) = {m1['assoc_B']:.2f}")
print(f"cut(A,B)   = {m1['cut_AB']:.2f}")
print(f"Ncut       = {m1['ncut']}")

### Fiedler vector visualization

The sign of each component in the Fiedler vector $\mathbf{y}_1$ determines the partition.  
Positive values → set $A$, negative values → set $B$.

In [ ]:
y1 = m1['fiedler_vector']
node_colors = ['#e67e22' if v >= 0 else '#27ae60' for v in y1]

plt.figure(figsize=(8, 4))
bars = plt.bar(range(len(y1)), y1, color=node_colors, edgecolor='black', linewidth=0.5)
plt.axhline(y=0, color='black', linewidth=1.2, linestyle='--')
plt.xticks(range(len(y1)), [f'node {i}' for i in range(len(y1))], rotation=45, fontsize=9)
plt.ylabel('Fiedler vector value', fontsize=11)
plt.title('Fiedler Vector — sign determines partition', fontsize=12)
plt.tight_layout()
plt.show()

### Example 2 — 8 nodes, 14 edges

In [ ]:
G2, W2, D2, pos2 = build_graph(N=8, M=14, seed=42)
A2, B2, m2 = normalized_cut(G2, W2, D2)

colors2 = ['#e67e22' if n in A2 else '#27ae60' for n in G2.nodes()]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
nx.draw(G2, pos2, with_labels=True, node_color='steelblue',
        edge_color='gray', node_size=600, font_color='white',
        font_weight='bold', ax=axes[0])
axes[0].set_title('Original Graph (N=8, M=14)', fontsize=11)

nx.draw(G2, pos2, with_labels=True, node_color=colors2,
        edge_color='gray', node_size=600, font_color='white',
        font_weight='bold', ax=axes[1])
axes[1].set_title('After Normalized Cut', fontsize=11)

plt.tight_layout()
plt.show()

print("Partition A (orange):", sorted(A2.tolist()))
print("Partition B (green) :", sorted(B2.tolist()))
print(f"Ncut = {m2['ncut']}")

---
## Part 3 — Recursive Normalized Cut

### Theory

A single Ncut gives a binary partition. For more than 2 groups, we apply Ncut **recursively** on each sub-partition, stopping when:
- A subgraph becomes too small (`min_size`), or
- The Ncut value exceeds a threshold (`max_ncut`) — meaning further splitting is not meaningful

This produces a hierarchical tree of partitions.

In [ ]:
def recursive_ncut(G, min_size=3, max_ncut=1.0, depth=0):
    """
    Recursively apply Normalized Cut until stopping criteria are met.

    Parameters
    ----------
    G        : networkx Graph (subgraph at current recursion level)
    min_size : int   — minimum allowed partition size
    max_ncut : float — maximum Ncut value to allow further splitting
    depth    : int   — current recursion depth (internal)

    Returns
    -------
    List of node lists — one per final partition
    """
    W = nx.to_numpy_array(G, weight='weight')
    degrees = W.sum(axis=1)

    # Guard: isolated nodes
    if np.any(degrees == 0):
        return [list(G.nodes())]

    D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees))
    L = np.diag(degrees) - W

    eigvals, eigvecs = eigh(L)
    z1 = eigvecs[:, 1]
    y1 = D_inv_sqrt @ z1

    partition = y1 >= 0
    set_A_idx = np.where(partition)[0]
    set_B_idx = np.where(~partition)[0]

    # Compute Ncut for this split
    assoc_A = np.sum(W[set_A_idx, :])
    assoc_B = np.sum(W[set_B_idx, :])
    cut_AB  = np.sum(W[np.ix_(set_A_idx, set_B_idx)])

    if assoc_A == 0 or assoc_B == 0:
        return [list(G.nodes())]

    ncut = (cut_AB / assoc_A) + (cut_AB / assoc_B)

    nodes = list(G.nodes())
    nodes_A = [nodes[i] for i in set_A_idx]
    nodes_B = [nodes[i] for i in set_B_idx]

    print(f"  [depth={depth}] |A|={len(nodes_A)}, |B|={len(nodes_B)}, "
          f"cut={cut_AB:.2f}, Ncut={ncut:.4f}")

    # Stop recursion after depth 0 if partition is too small or Ncut too high
    if depth > 0 and (len(set_A_idx) < min_size or len(set_B_idx) < min_size or ncut > max_ncut):
        return [list(G.nodes())]

    G_A = G.subgraph(nodes_A).copy()
    G_B = G.subgraph(nodes_B).copy()

    return recursive_ncut(G_A, min_size, max_ncut, depth + 1) + \
           recursive_ncut(G_B, min_size, max_ncut, depth + 1)

### Example — Recursive Ncut on 12 nodes, 22 edges

In [ ]:
G3, W3, D3, pos3 = build_graph(N=12, M=22, seed=30)

print("Recursive Ncut partitioning trace:")
partitions = recursive_ncut(G3, min_size=3, max_ncut=1.0)

print(f"\nFinal partitions ({len(partitions)} groups):")
for i, p in enumerate(partitions):
    print(f"  Partition {i}: nodes {sorted(p)}")

In [ ]:
tableau_colors = list(mcolors.TABLEAU_COLORS.values())
node_color_map = {}
for idx, part in enumerate(partitions):
    for node in part:
        node_color_map[node] = tableau_colors[idx % len(tableau_colors)]

color_list = [node_color_map[n] for n in G3.nodes()]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

nx.draw(G3, pos3, with_labels=True, node_color='steelblue',
        edge_color='gray', node_size=500, font_color='white',
        font_weight='bold', ax=axes[0])
axes[0].set_title('Original Graph (N=12, M=22)', fontsize=11)

nx.draw(G3, pos3, with_labels=True, node_color=color_list,
        edge_color='gray', node_size=500, font_color='white',
        font_weight='bold', ax=axes[1])
axes[1].set_title(f'Recursive Ncut — {len(partitions)} partitions', fontsize=11)

plt.tight_layout()
plt.show()

---
## Results & Metrics

### Ncut vs plain cut — comparison

Plain min-cut tends to isolate weakly connected nodes. Ncut penalizes this by normalizing against total connectivity. The table below shows the difference on the same graph.

In [ ]:
def compute_plain_cut(W, set_A, set_B):
    return float(np.sum(W[np.ix_(set_A, set_B)]))


print("=" * 55)
print(f"{'Metric':<25} {'N=12, M=21':>12}  {'N=8, M=14':>12}")
print("=" * 55)

for label, A_set, B_set, metrics in [
    ('N=12, M=21', A1, B1, m1),
    ('N=8,  M=14', A2, B2, m2)
]:
    print(f"\n{label}")
    print(f"  |A| = {len(A_set)}, |B| = {len(B_set)}")
    print(f"  cut(A,B)   = {metrics['cut_AB']:.2f}")
    print(f"  assoc(A,V) = {metrics['assoc_A']:.2f}")
    print(f"  assoc(B,V) = {metrics['assoc_B']:.2f}")
    print(f"  Ncut       = {metrics['ncut']}")
    balance = min(len(A_set), len(B_set)) / max(len(A_set), len(B_set))
    print(f"  Balance    = {balance:.2f}")

---
## Summary

| Property | Normalized Cut |
|----------|---------------|
| Input | Weighted graph |
| Core idea | Minimize Ncut via Fiedler vector of $L_N$ |
| Advantage over min-cut | Penalizes isolated small partitions |
| Relaxation | NP-hard → continuous via Rayleigh-Ritz quotient |
| Multi-way | Via recursive binary splitting |
| Key parameter | `max_ncut` threshold controls recursion depth |

**Next:** [Chapter 4 — Kernighan-Lin](04_kernighan_lin.ipynb)